# Atividade 4 — População 2010 x 2022 (Censo Demográfico)

Projeto que lê a tabela do IBGE (Censo Demográfico 2022 — População 2010 Compatibilizada) e:
- agrega a população por **estado (UF)**;
- calcula o crescimento populacional entre 2010 e 2022;
- repete a análise por **município**;
- salva as duas tabelas resultantes em CSV.


## 1. Importar bibliotecas

In [1]:
import pandas as pd

## 2. Ler a tabela

A planilha tem 2 linhas de título antes do cabeçalho de verdade, e algumas linhas de nota no rodapé — por isso usamos `skiprows=2` e depois removemos as linhas sem `COD. MUNIC` (notas de rodapé).

In [2]:
caminho_arquivo = "../dados/CD2022_Populacao_2010_Compatibilizada_20231222.xlsx"

df = pd.read_excel(caminho_arquivo, sheet_name="Municípios", skiprows=2)

# remove colunas/linhas totalmente vazias e linhas de nota de rodapé (sem código de município)
df = df.dropna(axis=1, how="all")
df = df.dropna(subset=["COD. MUNIC"])

df.head()

,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,População Município 2010\n(Sinopse),População 2010 (Alterações de Limites até 2022)1,População Censo 2022
0,RO,11.0,15.0,Alta Floresta D'Oeste,24392.0,24392.0,21494.0
1,RO,11.0,23.0,Ariquemes,90353.0,90353.0,96833.0
2,RO,11.0,31.0,Cabixi,6313.0,6313.0,5351.0
3,RO,11.0,49.0,Cacoal,78574.0,78574.0,86887.0
4,RO,11.0,56.0,Cerejeiras,17029.0,17029.0,15890.0


In [3]:
# Renomeando colunas para facilitar o uso
df = df.rename(columns={
    "População Município 2010\n(Sinopse)": "pop_2010_sinopse",
    "População 2010 (Alterações de Limites até 2022)1": "pop_2010_compatibilizada",
    "População Censo 2022": "pop_2022"
})

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5570 entries, 0 to 5569
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   UF                        5570 non-null   str    
 1   COD. UF                   5570 non-null   float64
 2   COD. MUNIC                5570 non-null   float64
 3   NOME DO MUNICÍPIO         5570 non-null   str    
 4   pop_2010_sinopse          5570 non-null   float64
 5   pop_2010_compatibilizada  5570 non-null   float64
 6   pop_2022                  5570 non-null   float64
dtypes: float64(5), str(2)
memory usage: 304.7 KB


## 3. Tabela agregada por estado (UF)

Somamos a população 2010 (compatibilizada, já ajustada por mudanças de limites territoriais) e 2022 de todos os municípios de cada UF.


In [4]:
pop_por_estado = (
    df.groupby("UF")[["pop_2010_compatibilizada", "pop_2022"]]
    .sum()
    .reset_index()
)

pop_por_estado.head()

,UF,pop_2010_compatibilizada,pop_2022
0,AC,733559.0,830018.0
1,AL,3120887.0,3127683.0
2,AM,3483985.0,3941613.0
3,AP,669526.0,733759.0
4,BA,14017071.0,14141626.0


## 4. Ordenar estados pelo crescimento populacional (2010 → 2022)

In [5]:
pop_por_estado["crescimento_absoluto"] = (
    pop_por_estado["pop_2022"] - pop_por_estado["pop_2010_compatibilizada"]
)
pop_por_estado["crescimento_percentual"] = (
    pop_por_estado["crescimento_absoluto"] / pop_por_estado["pop_2010_compatibilizada"] * 100
).round(2)

pop_por_estado = pop_por_estado.sort_values("crescimento_absoluto", ascending=False).reset_index(drop=True)
pop_por_estado

,UF,pop_2010_compatibilizada,pop_2022,crescimento_absoluto,crescimento_percentual
0,SP,41262199.0,44411238.0,3149039.0,7.63
1,SC,6248436.0,7610361.0,1361925.0,21.80
2,GO,6001789.0,7056495.0,1054706.0,17.57
3,PR,10444526.0,11444380.0,999854.0,9.57
4,MG,19597330.0,20539989.0,942659.0,4.81
5,MT,3035122.0,3658649.0,623527.0,20.54
6,PA,7581051.0,8120131.0,539080.0,7.11
7,AM,3483985.0,3941613.0,457628.0,13.14
8,CE,8451644.0,8794957.0,343313.0,4.06
9,ES,3514952.0,3833712.0,318760.0,9.07


## 5. Salvar a tabela por estado em CSV

In [6]:
pop_por_estado.to_csv(r"../saida/populacao_por_estado.csv", sep=";", index=False)
print("Salvo em ../saida/populacao_por_estado.csv")

Salvo em ../saida/populacao_por_estado.csv


## 6. Agora por município

Como cada linha da tabela original já é um município, não é preciso agregar — só selecionamos, calculamos o crescimento e ordenamos.


In [7]:
pop_por_municipio = df[
    ["UF", "COD. UF", "COD. MUNIC", "NOME DO MUNICÍPIO", "pop_2010_compatibilizada", "pop_2022"]
].copy()

pop_por_municipio["crescimento_absoluto"] = (
    pop_por_municipio["pop_2022"] - pop_por_municipio["pop_2010_compatibilizada"]
)
pop_por_municipio["crescimento_percentual"] = (
    pop_por_municipio["crescimento_absoluto"] / pop_por_municipio["pop_2010_compatibilizada"] * 100
).round(2)

pop_por_municipio = pop_por_municipio.sort_values("crescimento_absoluto", ascending=False).reset_index(drop=True)
pop_por_municipio.head(10)

,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,pop_2010_compatibilizada,pop_2022,crescimento_absoluto,crescimento_percentual
0,AM,13.0,2603.0,Manaus,1802014.0,2063689.0,261675.0,14.52
1,DF,53.0,108.0,Brasília,2572159.0,2817381.0,245222.0,9.53
2,SP,35.0,50308.0,São Paulo,11253503.0,11451999.0,198496.0,1.76
3,SP,35.0,52205.0,Sorocaba,586816.0,723682.0,136866.0,23.32
4,GO,52.0,8707.0,Goiânia,1301912.0,1437366.0,135454.0,10.40
5,RR,14.0,100.0,Boa Vista,284313.0,413486.0,129173.0,45.43
6,SC,42.0,5407.0,Florianópolis,421240.0,537211.0,115971.0,27.53
7,PA,15.0,5536.0,Parauapebas,153908.0,267836.0,113928.0,74.02
8,MS,50.0,2704.0,Campo Grande,786774.0,898100.0,111326.0,14.15
9,PB,25.0,7507.0,João Pessoa,723515.0,833932.0,110417.0,15.26


## 7. Salvar a tabela por município em CSV

In [8]:
pop_por_municipio.to_csv(r"../saida/populacao_por_municipio.csv", sep=";", index=False)
print("Salvo em ../saida/populacao_por_municipio.csv")

Salvo em ../saida/populacao_por_municipio.csv
